# **Notebook : Translation to R – Pseudobulk Aggregation for Differential Expression**

This notebook processes the preprocessed single-cell dataset (`adata_pp.h5ad`) from `single_cell_pipeline.ipynb`.

It performs pseudobulk aggregation (Macro-type × Donneur), applies statistical filters, and exports count matrices + metadata ready for differential expression analysis in R (limma/voom or DESeq2).

---

**Pipeline Overview:**
- Setup and Configuration
- Load Preprocessed Data and Validation
- Pseudobulk Aggregation (cell_type_annotation × donor_id)
- Statistical Filters (min cells per sample, min donors per cluster)
- Export for R Analysis
- Summary

<Small>## Pseudobulk aggregation par type cellulaire et donneur

### Objectif scientifique
Construire des profils d’expression pseudobulk à partir des données snRNA-seq HBCC,
en agrégeant les comptes d’expression au niveau **donneur × type cellulaire**.
Cette étape vise à définir des unités statistiques biologiquement indépendantes
(donneurs) afin d’éviter toute pseudo-réplication cellulaire dans les analyses
différentielles ultérieures.

### Logique méthodologique
Les données single-cell contiennent un grand nombre de noyaux par individu,
ce qui empêche l’utilisation directe de tests statistiques au niveau cellule.
Une approche pseudobulk consiste à sommer les comptes bruts de tous les noyaux
appartenant à un même donneur et à un même type cellulaire, produisant ainsi
une matrice comparable à des données bulk RNA-seq.

Aucune normalisation, transformation ou analyse différentielle n’est réalisée
à ce stade. Les annotations cellulaires de référence HBCC sont utilisées telles quelles,
sans clustering ni réannotation.

### Résultat attendu
- Une matrice de comptes pseudobulk (donneur × gènes) pour chaque type cellulaire
- Une table de métadonnées associée (diagnostic, identifiant donneur)
- Des objets prêts à être utilisés pour une analyse différentielle ultérieure
</Small>

# **Setup block**


### Setup and Configuration

In [1]:
# ==========================================================================================================
# GESTION SYSTÈME & ENVIRONNEMENT
import os                                       # Navigation fichiers (DIRS, chemins relatifs)
import gc                                       # Gestion mémoire (nettoyage objets inutilisés)
import json                                     # Lecture du dictionnaire de métadonnées (pipeline / EDA)  
import warnings                                 # Masquer warnings Scanpy/AnnData (dépréciation)
import re                                       # Parsing noms échantillons (regex pour donor_id)
from IPython.display import Markdown, display   # Affichage Jupyter (titres formatés, tableaux HTML)

# ----------------------------------------------------------------------------------------------------------
# CALCUL NUMÉRIQUE & VISUALISATION
# >> Générer les QC plots (n_genes vs n_counts, % mitochondrial, distributions par pathologie)
import numpy as np                              # Matrices creuses (X sparse), seuils QC (percentiles)
import math                                     # Calculs grilles subplots (ceil/floor pour layout)
import matplotlib.pyplot as plt                 # Figures principales (violin, scatter, hist)
from matplotlib.colors import to_rgb, rgb_to_hsv, hsv_to_rgb, rgb2hex
from matplotlib.gridspec import GridSpec
import seaborn as sns                           # Heatmaps corrélations, boxplots stylisés

# ----------------------------------------------------------------------------------------------------------
# ANALYSE SINGLE-CELL
# >> Charger données brutes (ad.read_h5ad), appliquer filtres QC (mitochondrial %, doublets),
#    calculer métriques (sc.pp.calculate_qc_metrics)
import anndata as ad                            # Objet AnnData (adata.obs, adata.var, adata.X)
import scanpy as sc                             # Fonctions QC (sc.pp.filter_cells, sc.pl.violin)
import pandas as pd                             # Métadonnées (fusion adata.obs), tableaux récapitulatifs

# ----------------------------------------------------------------------------------------------------------
# CLUSTERING (PRÉPARATION POUR S2 - NON UTILISÉ DANS S1)
# >> Importé par anticipation pour pipeline complet (normalisation → Leiden)
import leidenalg                                # Algorithme Leiden (clustering post-normalisation)
import igraph                                   # Graphe k-NN (requis pour leidenalg sous Scanpy)

# ==========================================================================================================
# --- DÉFINITION DES DOSSIERS ---
PROJECT_ROOT = "C:\\Z\\M2_AIDA\\transcriptomics_project"  # Laïla

DIRS = {
    "DATA":    os.path.join(PROJECT_ROOT, "data"),
    "EDA":     os.path.join(PROJECT_ROOT, "eda"),
    "FIGURES": os.path.join(PROJECT_ROOT, "figures"),
    "TMP":     os.path.join(PROJECT_ROOT, "tmp_cache")
}

for path in DIRS.values():
    os.makedirs(path, exist_ok=True)

os.chdir(PROJECT_ROOT)

# ==========================================================================================================
# --- PARAMÈTRES SCANPY ---
sc.settings.figdir = DIRS["FIGURES"]
sc.settings.cachedir = DIRS["TMP"]
sc.settings.datasetdir = DIRS["DATA"]
sc.settings.set_figure_params(dpi=100, frameon=False)

warnings.filterwarnings("ignore") 

# ==========================================================================================================
print(f"✅ Environnement chargé. Working directory: {os.getcwd()}")

✅ Environnement chargé. Working directory: c:\Z\M2_AIDA\transcriptomics_project


# **Data Loading**

### Dataset analytique normazlisé + UMAP exploitable pour DE :

<br>  local:       `data/HBCC_SCZ_CTRL_postQC_UMAP.h5ad`.

In [ ]:
dataset_path = os.path.join(DIRS["DATA"], "HBCC_SCZ_CTRL_postQC_UMAP.h5ad")
# adata = sc.read_h5ad(dataset_path) >>>>> MemoryError !
adata = sc.read_h5ad(dataset_path, backed='r')  # Lecture en mode backed pour économiser la mémoire
# ** Note technique :** backed mode pour éviter MemoryError.

dictionary_path = os.path.join(DIRS["DATA"],"hbcc_cellxgene_metadata_dictionary.json")
with open(dictionary_path, "r") as f:
    meta_hbcc = json.load(f)

print(f"✅ HBCC_SCZ_CTRL_postQC_UMAP and metadata dictionary successfully loaded (backed mode). Shape (cells, genes): {adata.shape}")

✅ HBCC_SCZ_CTRL_postQC_UMAP and metadata dictionary successfully loaded (backed mode). Shape (cells, genes): (209093, 34176)


## XXX

#### **Pseudobulk Aggregation by Cell Type and Donor**

<small>
Cette étape vise à résumer l’expression génique du cortex préfrontal humain en profils représentatifs par **donneur** et par **type cellulaire**, à partir des données single-cell de la cohorte HBCC. L’objectif est de disposer, pour chaque type cellulaire, d’une mesure globale d’expression comparable entre individus.  
<br><br>
L’agrégation par donneur permet de replacer l’analyse à l’échelle de l’individu, qui constitue l’unité d’intérêt pour la comparaison entre patients atteints de schizophrénie (SCZ) et sujets contrôles (CTRL). Les types cellulaires considérés correspondent aux annotations de référence fournies avec l’atlas HBCC.  
<br><br>
Cette étape produit des matrices d’expression synthétiques par type cellulaire, qui serviront de support aux analyses comparatives ultérieures entre groupes diagnostiques.
</small>


In [10]:
from scipy.sparse import csr_matrix

pb_matrices = {}
pb_metadata = {}

# Pour chaque type cellulaire
for cell_type in adata_mem.obs["cell_type"].unique():
    ad_ct = adata_mem[adata_mem.obs["cell_type"] == cell_type]

    # Initialisation de la somme pour chaque gène
    pb_counts = csr_matrix((1, ad_ct.n_vars))  # Matrice creuse vide pour les comptes par gène

    # Traitement des donneurs par lot pour éviter le MemoryError
    for donor_id in ad_ct.obs["donor_id"].unique():
        ad_donor = ad_ct[ad_ct.obs["donor_id"] == donor_id]

        # Ajout des comptes des cellules de ce donneur
        pb_counts += ad_donor.X.sum(axis=0)

    # Convertir en DataFrame sans conversion dense
    pb_counts_df = pd.DataFrame(
        pb_counts.T,  # Transposer pour avoir les gènes en lignes
        index=ad_ct.var_names,  # Noms des gènes
        columns=["Total_counts"]  # Une colonne par type cellulaire
    )

    # Métadonnées associées
    meta = ad_ct.obs[["donor_id", "disease"]].drop_duplicates()
    pb_metadata[cell_type] = meta

    pb_matrices[cell_type] = pb_counts_df

print(f"✅ Pseudobulk matrices generated for {len(pb_matrices)} cell types.")


✅ Pseudobulk matrices generated for 13 cell types.


#### **Interpretation of Pseudobulk Aggregation by Cell Type and Donor**

<small>
Cette étape permet d'agréger les comptes d'expression génique à l'échelle du **donneur** et **par type cellulaire**. L'objectif est de produire des profils d'expression globaux pour chaque type cellulaire en sommant les comptes de tous les noyaux d'un même donneur. Cela permet de traiter les données à un niveau **individuel**, ce qui est nécessaire pour les analyses différentielles ultérieures entre les groupes diagnostiques (schizophrénie vs contrôles).  
<br><br>
Chaque type cellulaire a été traité indépendamment, et les comptes ont été totalisés par gène pour chaque donneur. Ce processus produit des matrices d'expression pseudobulk prêtes pour l'analyse suivante. L'agrégation par type cellulaire a été réalisée de manière à réduire l'empreinte mémoire tout en maintenant une résolution suffisamment fine pour les analyses statistiques ultérieures.
</small>


#### **Differential Expression Analysis (SCZ vs CTRL) by Cell Type**

<small>
L’objectif de cette étape est d’identifier les gènes dont l’expression diffère de manière significative entre les groupes **schizophrénie (SCZ)** et **contrôles (CTRL)**, pour chaque **type cellulaire** dans la cohorte HBCC. L'analyse est réalisée à l’échelle du **donneur**, chaque donneur étant considéré comme une unité statistique. Cette approche permet de contrôler les variations inter-individus tout en analysant les différences au niveau des types cellulaires.

La méthode utilisée repose sur un modèle linéaire, où l’on teste les différences d’expression entre SCZ et CTRL, avec une correction pour les covariables pertinentes, telles que l'âge, le sexe, etc. Les résultats sont validés en contrôlant le taux de faux positifs par une correction FDR.

Les résultats de cette analyse fourniront une liste de gènes différentiellement exprimés, avec des informations sur l'effet estimé, les valeurs p et les valeurs ajustées.
</small>
